Data Preparation

In [1]:
import pandas as pd

df = pd.read_csv("data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")
df.shape

(7043, 21)

In [2]:
df.describe(include="all")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [3]:
from numpy import int32, int64


for col in df.columns:
    if df[col].nunique() == 2 and set(df[col].unique()) not in {0, 1}:
        df[col] = df[col].map({'Yes': 1,
                                 'No': 0,
                                 'Male': 1,
                                 'Female': 0,
                     }).fillna(0).astype(dtype=int64)
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,0,0,1,0,1,0,No phone service,DSL,No,...,No,No,No,No,Month-to-month,1,Electronic check,29.85,29.85,0
1,5575-GNVDE,1,0,0,0,34,1,No,DSL,Yes,...,Yes,No,No,No,One year,0,Mailed check,56.95,1889.5,0
2,3668-QPYBK,1,0,0,0,2,1,No,DSL,Yes,...,No,No,No,No,Month-to-month,1,Mailed check,53.85,108.15,1
3,7795-CFOCW,1,0,0,0,45,0,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,0,Bank transfer (automatic),42.30,1840.75,0
4,9237-HQITU,0,0,0,0,2,1,No,Fiber optic,No,...,No,No,No,No,Month-to-month,1,Electronic check,70.70,151.65,1


In [4]:
df = df.drop("customerID", axis=1)
df.head()
multi_cat_cols = [ col for col in df.columns if df[col].nunique() > 2 and df[col].nunique() < 5 ]
df = pd.get_dummies(df, columns=multi_cat_cols, drop_first=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 31 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   gender                                 7043 non-null   int64  
 1   SeniorCitizen                          7043 non-null   int64  
 2   Partner                                7043 non-null   int64  
 3   Dependents                             7043 non-null   int64  
 4   tenure                                 7043 non-null   int64  
 5   PhoneService                           7043 non-null   int64  
 6   PaperlessBilling                       7043 non-null   int64  
 7   MonthlyCharges                         7043 non-null   float64
 8   TotalCharges                           7043 non-null   object 
 9   Churn                                  7043 non-null   int64  
 10  MultipleLines_No phone service         7043 non-null   bool   
 11  Mult

In [5]:
bool_cols = df.select_dtypes(include=bool).columns
df[bool_cols] = df[bool_cols].astype(int64)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 31 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   gender                                 7043 non-null   int64  
 1   SeniorCitizen                          7043 non-null   int64  
 2   Partner                                7043 non-null   int64  
 3   Dependents                             7043 non-null   int64  
 4   tenure                                 7043 non-null   int64  
 5   PhoneService                           7043 non-null   int64  
 6   PaperlessBilling                       7043 non-null   int64  
 7   MonthlyCharges                         7043 non-null   float64
 8   TotalCharges                           7032 non-null   float64
 9   Churn                                  7043 non-null   int64  
 10  MultipleLines_No phone service         7043 non-null   int64  
 11  Mult

Model Training - Random Forest

In [6]:
import mlflow
import mlflow.models
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, recall_score
import sklearn.model_selection
import sklearn.metrics
import optuna
import time

from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score

#Initiate MLflow
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment(f"S25021301_COM763_{time.strftime("%Y-%m-%d %H:%M:%S")}")

#Extract the results (y) and split the data into test and train
y = df["Churn"]
X = df.drop("Churn", axis=1)

random_state = 36
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=random_state)

with mlflow.start_run(run_name="Random Forest"):
    #Define the Optuna objective method to optimize the hyperparameters
    def objective(trial: optuna.Trial):
        params = {
           "n_estimators": trial.suggest_int("n_estimators", 100, 600),
            "max_depth": trial.suggest_int("max_depth",3, 15),
            "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 5),
            "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
            "criterion": trial.suggest_categorical("criterion", ["gini", "entropy"]),
            "random_state": random_state,
            "n_jobs": -1, 
            "class_weight": trial.suggest_categorical("class_weight", ["balanced", "balanced_subsample"]) 
        }
        model = RandomForestClassifier(**params)
        scores = cross_val_score(model, X, y, cv=3, scoring="recall")
        return scores.mean()

    #Run 30 Optuna trials to identify the best hyperparameters
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=30)
    best_params = study.best_params

    #Train the model with the best params identified
    model = RandomForestClassifier(**best_params, random_state=36, n_jobs=-1)
    t_start = time.time()
    model.fit(X_train, y_train)
    t_taken = time.time() - t_start

    #Test the model
    p_start = time.time() 
    y_preds = model.predict(X_test)
    p_taken = time.time() - p_start

    #Measure Accuracy, Recall and F1-score
    accuracy = accuracy_score(y_test, y_preds)
    recall = recall_score(y_test, y_preds)
    f1 = f1_score(y_test, y_preds)

    #Save the model and document in mlflow
    mlflow.log_param("model", "Random Forest") 
    mlflow.log_params(best_params)
    mlflow.log_metric("train_time", t_taken)
    mlflow.log_metric("inference_time", p_taken)
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1-score", f1)
    mlflow.sklearn.log_model(model, name="model")


2026/07/05 09:02:59 INFO mlflow.tracking.fluent: Experiment with name 'S25021301_COM763_2026-07-05 09:02:59' does not exist. Creating a new experiment.
[I 2026-07-05 09:02:59,929] A new study created in memory with name: no-name-190d4b07-9e55-4015-85d6-b3f7daf757be
[I 2026-07-05 09:03:01,408] Trial 0 finished with value: 0.797752808988764 and parameters: {'n_estimators': 243, 'max_depth': 7, 'min_samples_split': 6, 'min_samples_leaf': 5, 'max_features': 'sqrt', 'criterion': 'gini', 'class_weight': 'balanced'}. Best is trial 0 with value: 0.797752808988764.
[I 2026-07-05 09:03:05,995] Trial 1 finished with value: 0.6923488496522204 and parameters: {'n_estimators': 414, 'max_depth': 12, 'min_samples_split': 8, 'min_samples_leaf': 4, 'max_features': None, 'criterion': 'entropy', 'class_weight': 'balanced_subsample'}. Best is trial 0 with value: 0.797752808988764.
[I 2026-07-05 09:03:11,314] Trial 2 finished with value: 0.830390583199572 and parameters: {'n_estimators': 397, 'max_depth': 3

🏃 View run Random Forest at: http://localhost:5000/#/experiments/6/runs/43045ea3dcab43889d44d913fcc62590
🧪 View experiment at: http://localhost:5000/#/experiments/6


Model Training - XGBoost

In [7]:
import mlflow.sklearn
import mlflow.xgboost
from xgboost import XGBClassifier

with mlflow.start_run(run_name="XGBoost"):
    #Define the Optuna objective method to fine-tune hyperparameters to XGBoost
    def objective(trial: optuna.Trial):
       params = {
            "n_estimators": trial.suggest_int("n_estimators", 300, 800),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2),
            "max_depth": trial.suggest_int("max_depth", 3, 10),
            "subsample": trial.suggest_float("subsample", 0.5, 1),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "scale_pos_weight": (y==0).sum() / (y==1).sum(),
            "random_state": random_state,
            "n_jobs": -1,
            "eval_metric": "logloss"
        } 
       
       model = XGBClassifier(**params)
       scores = cross_val_score(model, X_train, y_train, cv=3, scoring="recall")
       return scores.mean()

    study.optimize(objective, n_trials=30)
    best_params = study.best_params
    
    #Train the model with the best params identified
    model = XGBClassifier(**best_params, random_state=random_state, n_jobs=-1, eval_metric="logloss", scale_pos_weight=(y==0).sum() / (y==1).sum())
    t_start = time.time()
    model.fit(X_train, y_train)
    t_taken = time.time() - t_start

    #Test the model
    p_start = time.time() 
    y_preds = model.predict(X_test)
    p_taken = time.time() - p_start

    #Measure Accuracy, Recall and F1-score
    accuracy = accuracy_score(y_test, y_preds)
    recall = recall_score(y_test, y_preds)
    f1 = f1_score(y_test, y_preds)

    #Save the model and document in mlflow
    mlflow.log_param("model", "XGBoost") 
    mlflow.log_params(best_params)
    mlflow.log_metric("train_time", t_taken)
    mlflow.log_metric("inference_time", p_taken)
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1-score", f1)
    mlflow.xgboost.log_model(model, name="model")


[I 2026-07-05 09:04:40,865] Trial 30 finished with value: 0.669249298467804 and parameters: {'n_estimators': 658, 'learning_rate': 0.1769366203374122, 'max_depth': 3, 'subsample': 0.9195315788992416, 'colsample_bytree': 0.6383642489559119}. Best is trial 11 with value: 0.831995719636169.
[I 2026-07-05 09:04:42,584] Trial 31 finished with value: 0.772332847316085 and parameters: {'n_estimators': 778, 'learning_rate': 0.013139516962675526, 'max_depth': 3, 'subsample': 0.5286144452666733, 'colsample_bytree': 0.993431228080333}. Best is trial 11 with value: 0.831995719636169.
[I 2026-07-05 09:04:44,812] Trial 32 finished with value: 0.6607883412621548 and parameters: {'n_estimators': 661, 'learning_rate': 0.19847258680561758, 'max_depth': 3, 'subsample': 0.9075315214104893, 'colsample_bytree': 0.593672440386555}. Best is trial 11 with value: 0.831995719636169.
[I 2026-07-05 09:04:45,646] Trial 33 finished with value: 0.7069404682084411 and parameters: {'n_estimators': 391, 'learning_rate':

🏃 View run XGBoost at: http://localhost:5000/#/experiments/6/runs/19d9efe0b61e4a53982c5b92d410b8ed
🧪 View experiment at: http://localhost:5000/#/experiments/6


Model Training - LightGBM

In [8]:
from tabnanny import verbose

import lightgbm
import mlflow.lightgbm
import mlflow.lightgbm

with mlflow.start_run(run_name="LightGBM"):
    
    #Define the Optuna objective method to fine-tune hyperparameters to LightGBM
    def objective(trial: optuna.Trial):
       params = {
            "n_estimators": trial.suggest_int("n_estimators", 100, 300),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2),
            "max_depth": trial.suggest_int("max_depth", 3, 10),
            "subsample": trial.suggest_float("subsample", 0.5, 1),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "scale_pos_weight": (y==0).sum() / (y==1).sum(),
            "random_state": random_state,
            "n_jobs": -1,
            "objective": "binary",
            "metric": "binary_logloss",
            "verbose": -1 
        } 
       
       model = lightgbm.LGBMClassifier(**params)
       scores = cross_val_score(model, X_train, y_train, cv=3, scoring="recall")
       return scores.mean()

    study.optimize(objective, n_trials=30)
    best_params = study.best_params
    
    #Train the model with the best params identified
    model = lightgbm.LGBMClassifier(**best_params, n_jobs=-1, objective="binary", metric="binary_logloss", verbose=-1)
 
    t_start = time.time()
    model.fit(X_train, y_train)
    t_taken = time.time() - t_start

    #Test the model
    p_start = time.time() 
    y_preds = model.predict(X_test)
    p_taken = time.time() - p_start

    #Measure Accuracy, Recall and F1-score
    accuracy = accuracy_score(y_test, y_preds)
    recall = recall_score(y_test, y_preds)
    f1 = f1_score(y_test, y_preds)

    #Save the model and document in mlflow
    mlflow.log_param("model", "LightGBM") 
    mlflow.log_params(best_params)
    mlflow.log_metric("train_time", t_taken)
    mlflow.log_metric("inference_time", p_taken)
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1-score", f1)
    mlflow.lightgbm.log_model(model, name="model")

[I 2026-07-05 09:05:21,927] Trial 60 finished with value: 0.7654079884207278 and parameters: {'n_estimators': 250, 'learning_rate': 0.02536728130022324, 'max_depth': 4, 'subsample': 0.503962975706229, 'colsample_bytree': 0.9144035214001108}. Best is trial 11 with value: 0.831995719636169.
[I 2026-07-05 09:05:22,566] Trial 61 finished with value: 0.7253931595732981 and parameters: {'n_estimators': 274, 'learning_rate': 0.1867194429016918, 'max_depth': 3, 'subsample': 0.9498001408893346, 'colsample_bytree': 0.8426366081809119}. Best is trial 11 with value: 0.831995719636169.
[I 2026-07-05 09:05:23,143] Trial 62 finished with value: 0.7438493985093105 and parameters: {'n_estimators': 206, 'learning_rate': 0.13342449684866578, 'max_depth': 3, 'subsample': 0.5338041351290519, 'colsample_bytree': 0.9648606016323114}. Best is trial 11 with value: 0.831995719636169.
[I 2026-07-05 09:05:23,968] Trial 63 finished with value: 0.729244048062494 and parameters: {'n_estimators': 288, 'learning_rate'

🏃 View run LightGBM at: http://localhost:5000/#/experiments/6/runs/acd7553c34724a47b94e069dcff514de
🧪 View experiment at: http://localhost:5000/#/experiments/6
